# OpenRAW — unattended InvISP training on Kaggle

This one notebook works in two modes, picked automatically from what's attached (**Add Input**):

1. **Prepare** (nothing attached; set *Accelerator: None* — no GPU quota used): downloads the FiveK cameras below,
   preprocesses them into OpenRAW's compact training format within the 20 GB Kaggle keeps, and saves it.
   Then on this notebook's **Output** tab choose **New Dataset**.
2. **Train** (that dataset attached; set *Accelerator: GPU T4 x2* (one GPU is used)): trains until shortly before the 12 h session
   limit and stops cleanly. To continue next session, also attach **this notebook's own latest output** —
   it resumes exactly where it stopped. `STATUS.txt` in the output says *DONE* or *RUN AGAIN*.

Always start runs with **Save Version → Save & Run All (Commit)** so they run in the background.
Settings needed: **Internet on**, and a Kaggle **Secret** named `GITHUB_TOKEN` (the repo is private).
Full guide: `docs/kaggle.md` in the repo.

In [ ]:
import time
SESSION_START = time.time()   # for the 12 h session budget -- keep this the first line

# ---- configuration ---------------------------------------------------------
TASK = "openraw-fivek"        # checkpoint name; keep it the same across sessions
CAMERAS = "all"               # "all" (largest first, within the budget) or e.g. ["Nikon D700", "Canon EOS 5D"]
DATA_BUDGET_GB = 18           # Kaggle keeps 20 GB of output; leave room for checkpoints
EPOCHS = None                 # None = auto: ~195k steps, upstream InvISP's training budget
WORKERS = 3                   # data-loading processes (Kaggle VMs have 4 CPUs)
REPO_URL = "https://github.com/Swarce/OpenRAW.git"
BRANCH = "main"

In [ ]:
import os, subprocess, sys

# GitHub token from Kaggle Secrets (Add-ons -> Secrets -> GITHUB_TOKEN); never printed or saved
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret -- cloning without one (works only for a public repo):", type(e).__name__)

SRC = "/tmp/OpenRAW"
if not os.path.isdir(SRC):
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token or "\0", "***"))
    subprocess.run(["git", "-C", SRC, "remote", "set-url", "origin", REPO_URL])  # drop the token from .git/config
print(subprocess.run(["git", "-C", SRC, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Kaggle already has torch, numpy and pillow; add what OpenRAW's training needs (not OpenCV: it would clash)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rawpy", "colour-demosaicing", "imagecodecs", "tifffile"], check=True)

In [ ]:
sys.path.insert(0, SRC + "/kaggle")
import kaggle_runner as kr

result = kr.run(task=TASK, cameras=CAMERAS, budget_gb=DATA_BUDGET_GB, epochs=EPOCHS,
                workers=WORKERS, session_start=SESSION_START)
print("result:", result)